> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 5 · Notebook 08 — Sentiment, news and tail risk

**Sessions:** S13 (Market sentiment) · S14 (News & NLP sentiment) · S15 (Black swans & tail events) · [Lesson plan](../../docs/lessons/PART_05_ANALYTICS_LIBRARY.md) · graded labs in [`labs/part05/`](../../labs/part05/)

**You will:**
1. Join weekly sentiment data at the time it was **published**, not the date it describes.
2. Score headlines with a lexicon that understands "not".
3. Turn scored news into a decaying index.
4. Estimate a tail index and see normal VaR fail an exceedance test.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known structure, so you always know which effects are real and which are luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p5lib.py is in notebooks/part05/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p5lib as p

p.use_course_style()

## 1. Point-in-time: COT is measured Tuesday, published Friday

The CFTC's Commitments of Traders report describes positions on **Tuesday** and is released on **Friday at 15:30 New York**. A backtest that joins it on the Tuesday date knows it three days early. Each daily bar (stamped at the 16:00 close, in UTC) must get the latest release whose `available_at` is at or before the bar: `pd.merge_asof(..., left_on="ts", right_on="available_at", direction="backward")`.

In [ ]:
cot = p.cot_releases()
daily = p.daily_closes_utc()
cot.head(3)

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def pit_join(bars, releases, value_cols):
    r = releases.sort_values("available_at")[["available_at", *value_cols]]
    return pd.merge_asof(bars.sort_values("ts"), r, left_on="ts", right_on="available_at", direction="backward")

mine = p.attempt(pit_join, daily, cot, ["net_long"])
mine = p.check("pit_join", mine, p.pit_join(daily, cot, ["net_long"]))
mine.iloc[2:8]

In [ ]:
leaky = pd.merge_asof(daily, cot[["report_date", "net_long"]].rename(columns={"net_long": "leaky"}),
                      left_on="ts", right_on="report_date", direction="backward")
honest = p.pit_join(daily, cot, ["net_long"])
early = (leaky["leaky"] != honest["net_long"]) & honest["net_long"].notna()
print(f"joined on the report date, {early.mean():.0%} of bars see a number that was not public yet (Tuesday to Friday, every week)")

## 2. Headlines and negation

A lexicon scorer counts positive and negative words. Without negation, *"did not beat estimates"* scores as good news. Our rule: split the headline into **clauses** (at punctuation and "but"), and flip a word's sign when one of the previous `window` tokens **in the same clause** is a negator (`p.NEGATORS`). The score is the mean of the signs, 0 with no sentiment words.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def lexicon_score(text, positive=p.POSITIVE, negative=p.NEGATIVE, window=2):
    signs = []
    for clause in p.CLAUSE.split(text.lower()):
        toks = p.TOKEN.findall(clause)
        for i, t in enumerate(toks):
            s = 1 if t in positive else (-1 if t in negative else 0)
            if s == 0:
                continue
            if any(w in p.NEGATORS for w in toks[max(0, i - window):i]):
                s = -s
            signs.append(s)
    return float(sum(signs) / len(signs)) if signs else 0.0

mine = [lexicon_score(t) for t in p.HEADLINES]
mine = p.check("lexicon_score", mine, [p.lexicon_score(t) for t in p.HEADLINES])
naive = [p.lexicon_score(t, window=0) for t in p.HEADLINES]
pd.DataFrame({"headline": p.HEADLINES, "no negation": naive, "with negation": mine})

## 3. A decaying news index

Scored headlines arrive at random times. A usable feature is a **decay-weighted sum**: each item contributes `score · exp(−age/τ)` once it is published, and nothing before. With τ = 24 hours, yesterday's news counts about a third.

In [ ]:
rng = np.random.default_rng(4)
times = pd.Series(pd.to_datetime("2025-03-03 13:00", utc=True) + pd.to_timedelta(np.sort(rng.uniform(0, 10 * 24, 40)), unit="h"))
scores = rng.choice([-1.0, -0.5, 0.5, 1.0], size=40, p=[0.3, 0.2, 0.2, 0.3])
grid = pd.date_range("2025-03-03 12:00", periods=11 * 24, freq="1h", tz="UTC")
idx = p.decay_index(times, scores, grid, tau_hours=24)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(grid, idx, label="news index (τ = 24 h)")
ax.vlines(times, 0, scores, color=[p.PALETTE[2] if s > 0 else p.PALETTE[7] for s in scores], lw=1.5, label="scored headlines")
ax.axhline(0, color="black", lw=0.6); ax.legend(); ax.set_title("Each headline starts a decaying contribution when it is published"); plt.show()

## 4. Tails: how fat, and what it does to VaR

Returns have fat tails: big losses are far more common than a normal distribution allows. The **Hill estimator** measures the tail index ξ from the `k` largest losses (sorted descending, `x[0]` the biggest): `ξ = mean(ln(x[i] / x[k]))` for `i = 0 … k−1`. For Student-t returns with 3 degrees of freedom the true ξ is 1/3.

In [ ]:
r = p.fat_tailed_returns(5000, df_=3.0, seed=9)
losses = -r
print(f"daily σ {r.std():.2%}, worst day {losses.max():.2%} = {losses.max() / r.std():.1f} σ")

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def hill_estimator(losses, k):
    x = np.sort(np.asarray(losses, float))[::-1]  # largest first
    return float(np.mean(np.log(x[:k] / x[k])))

ks = [50, 100, 200, 400]
mine = [hill_estimator(losses, k) for k in ks]
mine = p.check("hill_estimator", mine, [p.hill_estimator(losses, k) for k in ks])
dict(zip(ks, np.round(mine, 3)))

The estimate depends on `k` (too few points: noisy; too many: the body of the distribution creeps in). Now the test that matters: fit VaR on the first half, count how often the second half's losses exceed it, and ask the **Kupiec** test whether that count is consistent with the promised rate.

In [ ]:
fit, test = losses[:2500], losses[2500:]
rows = []
for q in (0.99, 0.995, 0.999):
    z = {0.99: 2.326, 0.995: 2.576, 0.999: 3.090}[q]
    models = {"normal": z * fit.std(), "historical": np.quantile(fit, q), "Hill (EVT)": p.hill_var(fit, 100, q)}
    for name, var in models.items():
        x = int((test > var).sum())
        rows.append({"level": q, "model": name, "VaR": f"{var:.2%}", "expected": round(2500 * (1 - q), 1),
                     "exceedances": x, "Kupiec p": round(p.kupiec_pvalue(2500, x, 1 - q), 4)})
pd.DataFrame(rows)

Normal VaR looks fine at 99% and falls apart further out: at 99.9% the losses beyond it come several times more often than promised, and Kupiec rejects it. The tail-aware estimates stay close. That is the whole case for EVT in risk limits.

## Wrap-up

* Join every external dataset on **when it was known** (`available_at`), never on the date it describes.
* Negation and clauses matter even for simple lexicons; keep sentiment features point-in-time too.
* Use tail-aware VaR/ES and backtest exceedances.
* Graded version: `labs/part05/week20_sentiment_tail` (VIX regimes, breadth, COT join, lexicon, news de-duplication, Hill and GPD VaR/ES, Kupiec) and the Clinic W4 composite.